# Testing CuPy and Numba

In [ ]:
!nvidia-smi

In [ ]:
import sys
print("Current Kernel Python:", sys.executable)
print("\nLook-up paths:")
for path in sys.path:
    print(path)

## Testing CuPy

In [ ]:
#%pip install "cupy-cuda13x[ctk]"
%pip install --force-reinstall --no-cache-dir "cupy-cuda13x[ctk]"

In [ ]:
import cupy as cp

In [ ]:
x = cp.arange(6).reshape(2, 3).astype('f')

In [ ]:
x.sum(axis=1)

In [ ]:
%pip install "cupy-cuda12x[ctk]" # Check the CUDA installed with nvcc --version (that should match with the version provided by nvidia-smi)
%pip install numba
%pip install nvidia-cuda-nvcc-cu12
%pip install "cuda-toolkit[all]"


In [ ]:

import os

# Point Numba to the NVVM library inside your specific virtual environment
os.environ['NUMBA_NVVM_LIB_DIR'] = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages/nvidia/cuda_nvcc/nvvm/lib64'

import cupy as cp
from numba import cuda

# A simple Numba CUDA kernel
@cuda.jit
def add_one(x, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        out[i] = x[i] + 1

# 1. Create a CuPy array on the GPU
x_gpu = cp.arange(10)
out_gpu = cp.empty_like(x_gpu)

# 2. Call the Numba kernel with the CuPy arrays
add_one[1, 10](x_gpu, out_gpu)

# 3. Print the result
print("Input:", x_gpu)
print("Output:", out_gpu) 
# Expected Output: [ 1  2  3  4  5  6  7  8  9 10]

In [ ]:
import os

# 1. Point Numba to the base of the pip-installed NVCC toolchain
os.environ['CUDA_HOME'] = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages/nvidia/cuda_nvcc'

# 2. Import the packages AFTER setting the environment variable
import cupy as cp
from numba import cuda

# A simple Numba CUDA kernel
@cuda.jit
def add_one(x, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        out[i] = x[i] + 1

# 1. Create a CuPy array on the GPU
x_gpu = cp.arange(10)
out_gpu = cp.empty_like(x_gpu)

# 2. Call the Numba kernel with the CuPy arrays
add_one[1, 10](x_gpu, out_gpu)

# 3. Print the result
print("Input:", x_gpu)
print("Output:", out_gpu)

In [ ]:
import os
import sys

# 1. Map out the explicit paths inside your virtual environment
venv_base = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages'
nvvm_library_dir = os.path.join(venv_base, 'nvidia/cuda_nvcc/nvvm/lib64')
cuda_home_dir = os.path.join(venv_base, 'nvidia/cuda_nvcc')

# 2. Hard-set every path variable Numba and ctypes could possibly inspect
os.environ['NUMBA_NVVM_LIB_DIR'] = nvvm_library_dir
os.environ['CUDA_HOME'] = cuda_home_dir

if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = f"{nvvm_library_dir}:{os.environ['LD_LIBRARY_PATH']}"
else:
    os.environ['LD_LIBRARY_PATH'] = nvvm_library_dir

# 3. Now perform your imports clean from a fresh kernel
import cupy as cp
from numba import cuda

# 4. Define and test your Numba CUDA kernel
@cuda.jit
def add_one(x, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        out[i] = x[i] + 1

# Create arrays directly on your GTX 1650
x_gpu = cp.arange(10)
out_gpu = cp.empty_like(x_gpu)

# Execute the kernel
add_one[1, 10](x_gpu, out_gpu)

print("Input: ", x_gpu)
print("Output:", out_gpu)

In [ ]:
from numba import njit
import random

@njit
def monte_carlo_pi(nsamples):
    acc = 0
    for i in range(nsamples):
        x = random.random()
        y = random.random()
        if (x ** 2 + y ** 2) < 1.0:
            acc += 1
    return 4.0 * acc / nsamples

In [ ]:
monte_carlo_pi(10)

In [ ]:
import os
import glob

# Path to your virtual environment packages
nvidia_base_dir = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages/nvidia'
target_dir = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages/nvidia/cuda_nvcc/nvvm/lib64'

# 1. Search for any versioned libnvvm files inside the nvidia directory
search_pattern = os.path.join(nvidia_base_dir, '**', 'libnvvm.so*')
matches = glob.glob(search_pattern, recursive=True)

# Filter out any existing symlinks we might have checked
actual_files = [m for m in matches if not os.path.islink(m)]

if actual_files:
    real_library = actual_files[0]
    print(f" Found real NVVM library at: {real_library}")
    
    # 2. Define where Numba expects 'libnvvm.so'
    desired_symlink = os.path.join(target_dir, 'libnvvm.so')
    
    # 3. Create the symlink if it doesn't exist
    if not os.path.exists(desired_symlink):
        try:
            os.symlink(real_library, desired_symlink)
            print(f" Successfully created symlink:\n {desired_symlink} -> {real_library}")
        except Exception as e:
            print(f"❌ Failed creating symlink: {e}")
    else:
        print(f" Symlink already exists at: {desired_symlink}")
else:
    print("❌ Could not find any file matching 'libnvvm.so*' inside your pip packages.")

In [ ]:
import cupy as cp
from numba import cuda

@cuda.jit
def add_arrays_gpu(a, b, out):
    # Find the unique index of this specific GPU thread
    idx = cuda.grid(1)
    
    # Boundary check: Ensure the thread index doesn't overshoot the array size
    if idx < a.shape[0]:
        out[idx] = a[idx] + b[idx]  # No return! Writes directly to output array

# Create data directly on VRAM using CuPy
a_gpu = cp.arange(10)
b_gpu = cp.arange(10)
out_gpu = cp.empty_like(a_gpu)

# Launch: 1 block containing 10 threads
add_arrays_gpu[1, 10](a_gpu, b_gpu, out_gpu)

In [ ]:
import os
import sys

# 1. Define the virtual environment path rules
venv_packages = '/home/vruiz/envs/optical_flow_3D/lib/python3.14/site-packages'
nvvm_dir = os.path.join(venv_packages, 'nvidia/cuda_nvcc/nvvm/lib64')
cuda_home = os.path.join(venv_packages, 'nvidia/cuda_nvcc')

# 2. Apply patches BEFORE importing Numba or CuPy
os.environ['NUMBA_NVVM_LIB_DIR'] = nvvm_dir
os.environ['CUDA_HOME'] = cuda_home

if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = f"{nvvm_dir}:{os.environ['LD_LIBRARY_PATH']}"
else:
    os.environ['LD_LIBRARY_PATH'] = nvvm_dir

# 3. Now perform the GPU package imports safely
import cupy as cp
from numba import cuda

# 4. Define the GPU Vector Addition Kernel
@cuda.jit
def add_arrays_gpu(a, b, out):
    idx = cuda.grid(1)
    if idx < a.shape[0]:
        out[idx] = a[idx] + b[idx]

# 5. Create elements on the GPU via CuPy
a_gpu = cp.arange(10)
b_gpu = cp.arange(10)
out_gpu = cp.empty_like(a_gpu)

# 6. Launch across 1 block of 10 threads
add_arrays_gpu[1, 10](a_gpu, b_gpu, out_gpu)

# 7. Verify the output matches
print("Array A:", a_gpu)
print("Array B:", b_gpu)
print("Result :", out_gpu)

In [ ]:
%pip install tqdm

In [ ]:
import optical_flow_3D

In [ ]:
from optical_flow_3D import OF3D

In [ ]:
import numpy as np

In [ ]:
farneback = OF3D.Farneback3D(
    iters=5,
    num_levels=5,
    scale=0.5,
    spatial_size=7,
    presmoothing=7,
    filter_type="box",
    filter_size=21,
)

In [ ]:
depth = 64
height = 64
width = 64

vol_1 = np.random.rand(depth, height, width)
vol_2 = np.random.rand(depth, height, width)

In [ ]:
output_vz, output_vy, output_vx, output_confidence = farneback.calculate_flow(
    vol_1, vol_2, 
    start_point=(0, 0, 0),
    total_vol=(depth, height, width),
    sub_volume=(depth, height, width),
    overlap=(4, 4, 4),
    threadsperblock=(8, 8, 8),
)

In [ ]:
%pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
# 1. Select a slice to visualize (e.g., the middle slice along the z-axis)
slice_idx = depth // 2
vx_slice = output_vx[slice_idx, :, :]
vy_slice = output_vy[slice_idx, :, :]

In [ ]:
# 2. Downsample the data for better visualization
skip = 4
vx_downsampled = vx_slice[::skip, ::skip]
vy_downsampled = vy_slice[::skip, ::skip]

In [ ]:
# 3. Create the grid of coordinates for the arrows
x_coords, y_coords = np.meshgrid(
    np.arange(0, width, skip),
    np.arange(0, height, skip)
)

In [ ]:
# 4. Create the quiver plot
plt.figure(figsize=(8, 8))
plt.quiver(x_coords, y_coords, vx_downsampled, vy_downsampled, color='r', angles='xy', scale_units='xy', scale=1)
plt.title(f'2D Optical Flow Quiver Plot (Z-slice {slice_idx})')
plt.gca().invert_yaxis() # Invert y-axis to match image coordinates (origin at top-left)
plt.xlabel('Width (X-axis)')
plt.ylabel('Height (Y-axis)')
plt.grid()
plt.show()